In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-chroma langchain-openai langchain-classic

In [ ]:
!pip install pypdf

In [ ]:
!pip install langchain-core

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore


In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "Minha Chave de API"

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model_name= "gpt-4o-mini", max_tokens=500)

In [ ]:
#Carregar o PDF

from google.colab import files
uploaded = files.upload()

pdf_link = "DOC-SF238339076816-20230503.pdf"

loader = PyPDFLoader(pdf_link, extract_images= False)
pages = loader.load_and_split()

In [ ]:
# Splitter

child_splitter = RecursiveCharacterTextSplitter(chunk_size = 200)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 200,
    length_function = len,
    add_start_index = True
)

In [ ]:
#Storages
store = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings, persist_directory='childVectorDB')

In [ ]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_document_retriever.add_documents(pages, ids=None)

In [ ]:
parent_document_retriever.vectorstore.get()

In [ ]:
TEMPLATE = """
  Você é um especialista em legislação e tecnologia. Responda a pergunta abaixo utilizando o contexto informado.
  Query:
  {question}

  Context:
  {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [ ]:
setup_retrival = RunnableParallel({"question": RunnablePassthrough(), "context": parent_document_retriever})

output_parser = StrOutputParser()

In [ ]:
parent_chain_retrival = setup_retrival | rag_prompt | llm | output_parser

In [ ]:
parent_chain_retrival.invoke("Quais os principais riscos do marco legal de ia?")

'O marco legal da inteligência artificial (IA) traz consigo diversas considerações e riscos que devem ser cuidadosamente analisados. A seguir, estão listados os principais riscos associados a este novo arcabouço regulatório, conforme o contexto apresentado:\n\n1. **Avaliação de Riscos**: A necessidade de categorização e avaliação de riesgos implica que diferentes aplicações de IA serão tratadas de maneiras distintas, o que pode criar lacunas de proteção para sistemas considerados de baixo risco, mas que, na prática, possam causar danos significativos. \n\n2. **Responsabilidade Civil**: A distinção na responsabilidade civil entre sistemas de alto risco e aqueles que não são classificados dessa forma pode gerar incertezas legais. A inversão do ônus da prova em favor da vítima para sistemas que não sejam de alto risco pode resultar em desafios na prática de responsabilização de agentes causadores de danos.\n\n3. **Interpretação e Implementação das Normas**: A implementação das regras e pr